Необходимо:

Определить топ-5 сценаристов по количеству фильмов в подборке.
Если несколько сценаристов имеют одинаковое количество фильмов, они упорядочиваются в лексикографическом порядке строк Python.
Для каждого из этих сценаристов выбрать самый популярный фильм, то есть фильм с лучшим местом в рейтинге (минимальное значение rating).
Посчитать сумму рейтингов (rating_ball) этих пяти фильмов.
Обращаем внимание:

Можно использовать функцию split_col (см. примечания).
Каждый сценарист учитывается отдельно, даже если в одном фильме указано несколько сценаристов.
Для каждого сценариста берётся ровно один фильм — его самый популярный.
Формат вывода
Сумма рейтингов топ-5 фильмов сценаристов, округленная до 3-х знаков после запятой

Примечания
Функция split_col:
```python
def split_col(s):
    s = s.fillna("")
    s = s.astype(str)
    s = s.str.split(";")
    s = s.apply(lambda lst: [x.strip() for x in lst if x.strip()])
    return s 
```

In [2]:
import pandas as pd

df = pd.read_csv('../kinopoisk-top250.csv')

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rating        250 non-null    int64  
 1   movie         250 non-null    str    
 2   year          250 non-null    int64  
 3   country       250 non-null    str    
 4   rating_ball   250 non-null    float64
 5   overview      250 non-null    str    
 6   director      250 non-null    str    
 7   screenwriter  250 non-null    str    
 8   actors        250 non-null    str    
 9   url_logo      250 non-null    str    
dtypes: float64(1), int64(2), str(7)
memory usage: 19.7 KB
None


In [ ]:
# screenwriters = df['screenwriter'].value_counts()

screenwriters = df['screenwriter'].apply(lambda x: x.split('; '))
display(screenwriters)

0                        [ Фрэнк Дарабонт,  Стивен Кинг]
1                        [ Фрэнк Дарабонт,  Стивен Кинг]
2                             [ Эрик Рот,  Уинстон Грум]
3                      [ Стивен Зеллиан,  Томас Кенилли]
4      [ Оливье Накаш,  Филипп Поццо ди Борго,  Эрик ...
                             ...                        
245                     [ Роберт Бентон,  Эйвери Кормэн]
246     [ Тед Эллиот,  Терри Россио,  Стюарт Битти, ...]
247                      [ Алесь Адамович,  Элем Климов]
248    [ Мишель Одиар,  Патрик Александер,  Жорж Лотн...
249                                  [ Эдгар Дубровский]
Name: screenwriter, Length: 250, dtype: object

In [10]:
from collections import defaultdict

# Инициализируем defaultdict с типом int (счётчик)
dict_of_actors = defaultdict(int)

def get_actors(col, res=dict_of_actors):
    actors = col.split('; ')
    if actors and actors[-1] == '...':
        actors.pop()  # Удаляем '...' если есть
    for actor in actors:
        res[actor] += 1  # Теперь работает: int по умолчанию = 0
    return len(actors)

# Применяем функцию
df['actors_count'] = df['actors'].apply(get_actors, res=dict_of_actors)

# Выводим результат
print(dict(dict_of_actors))  # Преобразуем в обычный dict для печати
print(df[['actors_count', 'actors']])

{'Тим Роббинс': 1, 'Морган Фриман': 5, 'Боб Гантон': 1, 'Уильям Сэдлер': 1, 'Клэнси Браун': 1, 'Том Хэнкс': 7, 'Дэвид Морс': 1, 'Бонни Хант': 1, 'Майкл Кларк Дункан': 1, 'Джеймс Кромуэлл': 2, 'Робин Райт': 1, 'Салли Филд': 1, 'Гэри Синиз': 1, 'Майкелти Уильямсон': 1, 'Лиам Нисон': 1, 'Бен Кингсли': 4, 'Рэйф Файнс': 1, 'Кэролайн Гудолл': 1, 'Эмбет Дэвидц': 1, 'Франсуа Клюзе': 1, 'Омар Си': 1, 'Анн Ле Ни': 1, 'Одри Флеро': 1, 'Жозефин де Мо': 1, 'Леонардо ДиКаприо': 6, 'Джозеф Гордон-Левитт': 2, 'Эллен Пейдж': 1, 'Том Харди': 3, 'Кэн Ватанабэ': 2, 'Жан Рено': 1, 'Гари Олдман': 4, 'Натали Портман': 2, 'Дэнни Айелло': 1, 'Питер Эппел': 1, 'Мэттью Бродерик': 1, 'Джереми Айронс': 1, 'Нэйтан Лейн': 1, 'Эрни Сабелла': 1, 'Джеймс Эрл Джонс': 1, 'Эдвард Нортон': 3, 'Брэд Питт': 6, 'Хелена Бонем Картер': 4, 'Мит Лоаф': 1, 'Зэк Гренье': 1, 'Александр Демьяненко': 3, 'Юрий Яковлев': 2, 'Леонид Куравлёв': 2, 'Наталья Крачковская': 2, 'Савелий Крамаров': 2, 'Роберто Бениньи': 1, 'Николетта Браски': 1

In [11]:
# Подсчитываем количество фильмов для каждого режиссера
director_counts = df['director'].value_counts()

# Выбираем топ-5 режиссеров
top_directors = director_counts.head(5)

# Формируем строку с фамилиями через запятую
result = ','.join(top_directors.index)

# Выводим результат
print(result)

 Игорь Масленников, Роберт Земекис, Стивен Спилберг, Кристофер Нолан, Дэвид Финчер


In [14]:
# Очистка данных: удаляем лишние пробелы и оставляем только фамилии
df['director'] = df['director'].str.strip()  # Удаляем пробелы в начале и конце
df['director'] = df['director'].str.split().str[-1]  # Оставляем только последнее слово (фамилию)

# Подсчитываем количество фильмов для каждого режиссера
director_counts = df['director'].value_counts()

df['director'] = df['director'].str.split(',')
df = df.explode('director')  # Создаем отдельные строки для каждого режиссера

# Остальной код остается таким же
df['director'] = df['director'].str.strip()
df['director'] = df['director'].str.split().str[-1]
director_counts = df['director'].value_counts()
director_df = director_counts.reset_index()
director_df.columns = ['director', 'count']
director_df = director_df.sort_values(by=['count', 'director'], ascending=[False, True])
top_directors = director_df.head(5)
result = ','.join(top_directors['director'])
print(result)

Масленников,Джексон,Гайдай,Земекис,Нолан
